# Aff-Wild2 Stage 1 — visual feature extraction

Runs `extract_visual_features` across the 307 video folders in `../competition-data/cropped_aligned/` for both EmotiEffLib backbones. The resulting `.npz` caches feed `aw2_02_stage1_train_eval.ipynb`.

Time budget: ~30–60 min per backbone on a consumer GPU, several hours on CPU. The extractor skips videos whose `.npz` already exists, so re-running is cheap.

In [1]:
import os, sys
from pathlib import Path

REPO = Path.cwd().resolve().parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

COMP_DATA = (REPO.parent / 'competition-data').resolve()
CROPPED_ALIGNED = COMP_DATA / 'cropped_aligned'
CACHE_ROOT = REPO / 'cache' / 'features'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
print('cropped_aligned :', CROPPED_ALIGNED)
print('cache root      :', CACHE_ROOT)

cropped_aligned : C:\Users\Andrey Lyaschenko\Documents\vkr\competition-data\cropped_aligned
cache root      : C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\cache\features


In [2]:
import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device name  :', torch.cuda.get_device_name(0))

cuda available: True
device name  : NVIDIA GeForce RTX 3050 Ti Laptop GPU


## 1. `enet_b0_8_va_mtl` — primary backbone (1280-D)

Expected cache size on disk: $\sim$0.9 GB across 307 `.npz` files.

In [3]:
from src.features.extract_visual import extract_visual_features

ENET_CACHE = CACHE_ROOT / 'enet_b0_8_va_mtl'
extract_visual_features(
    model_name='enet_b0_8_va_mtl',
    cropped_aligned_dir=CROPPED_ALIGNED,
    output_dir=ENET_CACHE,
    engine='torch',
    batch_size=48,
    device=None,
    overwrite=False,
)

c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
extract[enet_b0_8_va_mtl]: 100%|██████████| 307/307 [56:23<00:00, 11.02s/it]  


In [5]:
import numpy as np

npz_files = sorted(ENET_CACHE.glob('*.npz'))
print('enet cache files :', len(npz_files))
assert len(npz_files) == 307, f'expected 307 caches, got {len(npz_files)}'

sample = np.load(npz_files[0], allow_pickle=True)
print('keys         :', list(sample.keys()))
print('features     :', sample['features'].shape, sample['features'].dtype)
print('scores       :', sample['scores'].shape, sample['scores'].dtype)
print('image_names  :', sample['image_names'].shape, '->', sample['image_names'][0])
assert sample['features'].shape[1] == 1280
assert sample['scores'].shape[1] == 10
total = sum(np.load(p)['features'].shape[0] for p in npz_files)
print(f'total frames cached: {total:,}')

enet cache files : 307
keys         : ['features', 'scores', 'image_names']
features     : (363, 1280) float32
scores       : (363, 10) float32
image_names  : (363,) -> 1-30-1280x720/00001.jpg
total frames cached: 168,097


## 2. `mbf_va_mtl` — secondary backbone (512-D)

In [ ]:
MBF_CACHE = CACHE_ROOT / 'mbf_va_mtl'
extract_visual_features(
    model_name='mbf_va_mtl',
    cropped_aligned_dir=CROPPED_ALIGNED,
    output_dir=MBF_CACHE,
    engine='torch',
    batch_size=48,
    device=None,
    overwrite=False,
)

In [ ]:
npz_files = sorted(MBF_CACHE.glob('*.npz'))
print('mbf cache files :', len(npz_files))
assert len(npz_files) == 307

sample = np.load(npz_files[0], allow_pickle=True)
print('features     :', sample['features'].shape)
print('scores       :', sample['scores'].shape)
assert sample['features'].shape[1] == 512
assert sample['scores'].shape[1] == 10
total = sum(np.load(p)['features'].shape[0] for p in npz_files)
print(f'total frames cached: {total:,}')

## 3. Annotation coverage check

Cross-check every annotation row against the caches via `features_index`. `num_missed` should be zero or close to it — a non-zero count means some videos are annotated but not extracted (typically a naming mismatch).

In [ ]:
from src.datasets.affwild2_mtl import read_mtl_annotations
from src.utils.io import load_features_dir

TRAIN_ANN = COMP_DATA / 'training_set_annotations.txt'
VAL_ANN   = COMP_DATA / 'validation_set_annotations.txt'

for cache, tag in [(ENET_CACHE, 'enet'), (MBF_CACHE, 'mbf')]:
    print(f'--- {tag} ({cache}) ---')
    filename2features, _sizes = load_features_dir(cache)
    for split_name, path in [('train', TRAIN_ANN), ('val', VAL_ANN)]:
        a = read_mtl_annotations(path, features_index=filename2features)
        print(f'  {split_name:>5}: kept={len(a):,}  missed={a.num_missed}')

If both caches cover every row (`missed=0`), proceed to `aw2_02_stage1_train_eval.ipynb`.